In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# fusion method 비교: CUE(CUE, softmax fusion) vs SUM(단순 합산, weight=1,1) vs
# Concat (Early)(cue_concat: 모달리티별 MLP 없이 원본 feature를 concat 후 downstream MLP로 직접 입력) vs
# Concat (Int)(cmfe: 모달리티별 MLP를 거친 embedding을 concat, fusion_policy="concat").
# 네 결과 모두 동일한 5-fold 내부 CV 파이프라인에서 나온 것이며, 유일한 차이는 fusion 방식이다.
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
cue_concat_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue_concat/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 네 가지 소스: (표시 이름, metrics.csv 루트 경로)
model_sources = {
    'SUM': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'SUM'),
    'Concat (Early)': os.path.join(cue_concat_base, '{dataset}', '{target}', 'v1', 'FP+IMG'),
    'Concat (Int)': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CONCAT'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
}

# CUE가 항상 마지막에 오도록 순서를 고정한다.
model_order = ['SUM', 'Concat (Early)', 'Concat (Int)', 'CUE']


In [2]:
def load_metric(model_dir_template, dataset, target):
    """cue/cue_concat의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    두 저장소 모두 동일한 aggregate_metrics 함수를 사용하므로 스키마가 동일하다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")


✅ 데이터 로드 완료: 32개의 항목이 수집되었습니다.


In [3]:
def plot_comparison_styled(df_plot, dataset, target_name, metric_name, output_dir):
    """지표 라벨은 지우되, Y축 눈금 수치와 배경은 유지한 버전.
    SUM/Concat (Early)/Concat (Int)/CUE를 해치 없이 colors 색상으로만 구분한다 (CUE가 항상 마지막 막대).
    막대 높이도 텍스트 라벨과 동일하게 소수점 셋째 자리까지 반올림한 값을 사용한다."""
    df_target = df_plot.set_index('Model').reindex(model_order).dropna(subset=[metric_name]).reset_index()
    if df_target.empty: return

    models = df_target['Model'].tolist()
    values = [round(v, 3) for v in df_target[metric_name].tolist()]
    # model_order(SUM, Concat (Early), Concat (Int), CUE) 순서에 맞춰 4색을 배정한다.
    # CUE는 항상 마지막이므로 weight_Plot.ipynb와 동일하게 06d6a0으로 통일한다.
    colors = ['#ef476f', '#ffd166', '#118ab2', '#06d6a0']

    y_min, y_max = min(values), max(values)
    margin = (y_max - y_min) * 0.45 if (y_max - y_min) != 0 else 0.1

    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(models))

    bars = ax.bar(x, values, color=colors[:len(models)], edgecolor='black', alpha=0.9, linewidth=1)

    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + (margin * 0.03),
                f'{height:.3f}', ha='center', va='bottom', fontsize=15,
                fontweight='bold')

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])

    ax.tick_params(axis='y', labelsize=11)
    ax.tick_params(axis='x', which='both', length=0)

    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)

    ax.set_ylim(max(0, y_min - (margin*0.1)), y_max + margin)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")


In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'fusion_comparison_v2')

for d_name in datasets:
    for target in targets:
        df_target = df_total[(df_total['Dataset'] == d_name) & (df_total['Target'] == target)]
        if df_target.empty: continue

        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            if m in df_target.columns:
                plot_comparison_styled(df_target, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/DIS/PI_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/DIS/PI_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/DIS/PI_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/DIS/Ssel_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/DIS/Ssel_pcc.png


... (8 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/Ssel_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/Ssel_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/WS(alpha=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/WS(alpha=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/WS(alpha=2)_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/RS(k=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/RS(k=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/fusion_comparison_v2/INH/RS(k=2)_pcc.png
